# Cross-City Air Quality Prediction — Indian Cities

### Experimental design

**Training cities:** Delhi, Mumbai, Kolkata, Chennai  
**Completely unseen test cities:** Bengaluru, Hyderabad

This notebook does **not** use a random 80/20 split across all cities. The test cities are kept completely separate.

**Target:** next-hour PM2.5  
**Input:** previous 24 hourly observations of PM2.5 + common pollutant variables + time features  
**Main model:** CNN-LSTM hybrid  
**Baselines:** Linear Regression, Random Forest, XGBoost, LSTM, GRU, CNN-LSTM  
**Metrics:** MAE, RMSE, R², MAPE

> Important: Download the CSV files from the linked CPCB/OpenCity dataset pages and place them in the folders shown below. The loader reads every CSV inside each city folder and attempts to standardize common CPCB column names.


## Dataset links

The training data below are CPCB-sourced hourly station datasets hosted by OpenCity:

- **Delhi:** https://data.opencity.in/dataset/delhi-hourly-air-quality-reports
- **Mumbai:** https://data.opencity.in/dataset/mumbai-hourly-air-quality-reports
- **Kolkata:** https://www.data.opencity.in/dataset/kolkata-hourly-air-quality-reports
- **Chennai:** https://data.opencity.in/dataset/chennai-hourly-air-quality-reports

OpenCity currently describes these as hourly CPCB datasets: Delhi has 39 stations (2017–2023), Mumbai stations from 2017–2023, Chennai 9 stations (2017–2023), and Kolkata 7 stations. citeturn1search2turn1search0turn1search1

For the **unseen test cities**, this notebook supports the CPCB-derived Indian-city dataset:
- **Bengaluru / Hyderabad:** https://www.kaggle.com/datasets/rajanbhateja/indian-cities-aqi-2020-2024

That dataset lists Bengaluru and Hyderabad locations and states that the data were collected from CPCB. citeturn0search11

### Folder structure

```text
air_quality_project/
│
├── data/
│   ├── train/
│   │   ├── Delhi/
│   │   ├── Mumbai/
│   │   ├── Kolkata/
│   │   └── Chennai/
│   │
│   └── test/
│       ├── Bengaluru/
│       └── Hyderabad/
│
└── cross_city_air_quality_prediction.ipynb
```

Download the CSV resources for each training city from its dataset page. For Mumbai, for example, the OpenCity page exposes hourly station CSV resources and identifies CPCB/airquality.cpcb.gov.in as the source. citeturn1search0turn1search7


In [ ]:
# Install dependencies
!pip -q install pandas numpy scikit-learn xgboost tensorflow matplotlib seaborn joblib


In [ ]:
import os
import glob
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import joblib

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Dense, LSTM, GRU, Conv1D, MaxPooling1D,
    Dropout, Bidirectional
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)


In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = "data"

TRAIN_CITIES = ["Delhi", "Mumbai", "Kolkata", "Chennai"]
TEST_CITIES = ["Bengaluru", "Hyderabad"]

LOOKBACK = 24       # previous 24 hours
HORIZON = 1         # predict PM2.5 one hour ahead

os.makedirs(BASE_DIR, exist_ok=True)

for city in TRAIN_CITIES:
    os.makedirs(os.path.join(BASE_DIR, "train", city), exist_ok=True)
for city in TEST_CITIES:
    os.makedirs(os.path.join(BASE_DIR, "test", city), exist_ok=True)

print("Put downloaded CSV files into:")
for city in TRAIN_CITIES + TEST_CITIES:
    split = "train" if city in TRAIN_CITIES else "test"
    print(f"  {BASE_DIR}/{split}/{city}/")


In [ ]:
def parse_opencity_csv(filepath):
    """
    Parse an OpenCity wide-format PM2.5 CSV into a long-format DataFrame.

    Format:
        Year,2017
        January-2017,00:00:00,01:00:00,...,23:00:00
        1,97.0,82.0,...
        2,55.0,...
        \\n   <- separator rows skipped
        Year,2018
        ...
    Returns DataFrame with columns: datetime (Timestamp), pm25 (float).
    """
    records = []
    current_month_date = None
    hours = []

    with open(filepath, "r", encoding="utf-8", errors="replace") as fh:
        for line in fh:
            line = line.rstrip("\n").rstrip("\r")
            parts = [p.strip() for p in line.split(",")]

            # Skip "Year,YYYY" header rows
            if parts[0] == "Year":
                continue

            # Skip blank / separator rows
            if parts[0] in ("", "\\n", r"\n"):
                continue

            # Month-header row: "January-2017", "February-2018", ...
            if re.match(r"[A-Za-z]+-\d{4}$", parts[0]):
                try:
                    current_month_date = pd.to_datetime(parts[0], format="%B-%Y")
                    hours = parts[1:]   # ["00:00:00", "01:00:00", ...]
                except Exception:
                    current_month_date = None
                continue

            # Data row: day_of_month, val_h0, val_h1, ..., val_h23
            if current_month_date is None:
                continue
            try:
                day = int(parts[0])
            except (ValueError, IndexError):
                continue

            for h_idx, h_str in enumerate(hours):
                col_idx = h_idx + 1
                val_str = parts[col_idx] if col_idx < len(parts) else ""
                try:
                    pm25_val = float(val_str)
                except (ValueError, TypeError):
                    pm25_val = np.nan

                try:
                    hour = pd.to_datetime(h_str, format="%H:%M:%S").hour
                    dt = pd.Timestamp(
                        year=current_month_date.year,
                        month=current_month_date.month,
                        day=day,
                        hour=hour,
                    )
                    records.append({"datetime": dt, "pm25": pm25_val})
                except Exception:
                    continue

    df = pd.DataFrame(records)
    if df.empty:
        raise ValueError(f"No data could be parsed from {filepath}")
    return df


def read_city_folder(folder, city):
    """Load all CSVs in a city folder and concatenate them."""
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not files:
        raise FileNotFoundError(
            f"No CSV files found in {folder}. Download city data first."
        )

    frames = []
    print(f"\n{city}: {len(files)} CSV file(s)")
    for f in files:
        try:
            df = parse_opencity_csv(f)
            df["station_name"] = os.path.splitext(os.path.basename(f))[0]
            df["city"] = city
            frames.append(df)
            print(f"  loaded: {os.path.basename(f)}  {df.shape}")
        except Exception as e:
            print(f"  SKIPPED: {os.path.basename(f)} | {e}")

    if not frames:
        raise ValueError(f"Could not read any usable CSV files in {folder}")
    return pd.concat(frames, ignore_index=True)


def load_split(cities, split):
    all_frames = []
    for city in cities:
        folder = os.path.join(BASE_DIR, split, city)
        all_frames.append(read_city_folder(folder, city))
    return pd.concat(all_frames, ignore_index=True)


train_raw = load_split(TRAIN_CITIES, "train")
test_raw  = load_split(TEST_CITIES,  "test")

print("\nTrain raw shape:", train_raw.shape)
print("Test raw shape :", test_raw.shape)

In [ ]:
# Inspect columns before preprocessing
print("TRAIN columns:")
print(sorted(train_raw.columns.tolist()))

print("\nTEST columns:")
print(sorted(test_raw.columns.tolist()))

print("\nTraining cities:")
print(train_raw["city"].value_counts())

print("\nTesting cities:")
print(test_raw["city"].value_counts())


In [ ]:
def prepare_city_data(df):
    df = df.copy()

    # Ensure correct dtypes
    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df["pm25"]     = pd.to_numeric(df["pm25"], errors="coerce")

    # Drop physically impossible values
    df.loc[df["pm25"] < 0,    "pm25"] = np.nan
    df.loc[df["pm25"] > 1000, "pm25"] = np.nan

    df = df.dropna(subset=["datetime", "pm25"]).copy()
    df = df.sort_values(["city", "station_name", "datetime"]).reset_index(drop=True)
    return df


train = prepare_city_data(train_raw)
test  = prepare_city_data(test_raw)

print("Train after cleaning:", train.shape)
print("Test  after cleaning :", test.shape)

## Important preprocessing decision

For cross-city generalization, we will **not randomly shuffle observations**.

Each city is processed chronologically. The model is trained only on the training cities. Bengaluru and Hyderabad are never used for fitting scalers, model weights, feature selection, or hyperparameter tuning.

Missing values are interpolated only inside each city/station time series, then remaining missing rows are dropped.


In [ ]:
def add_time_features(df):
    df = df.copy()
    dt = df["datetime"]

    df["hour_sin"]  = np.sin(2 * np.pi * dt.dt.hour / 24)
    df["hour_cos"]  = np.cos(2 * np.pi * dt.dt.hour / 24)

    df["dow_sin"]   = np.sin(2 * np.pi * dt.dt.dayofweek / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * dt.dt.dayofweek / 7)

    df["month_sin"] = np.sin(2 * np.pi * (dt.dt.month - 1) / 12)
    df["month_cos"] = np.cos(2 * np.pi * (dt.dt.month - 1) / 12)

    return df


train = add_time_features(train)
test  = add_time_features(test)

# Only PM2.5 + cyclic time features are available in these CSVs
FEATURES = [
    "pm25",
    "hour_sin", "hour_cos",
    "dow_sin",  "dow_cos",
    "month_sin", "month_cos",
]

# Keep only features present in both splits (future-proof)
FEATURES = [f for f in FEATURES if f in train.columns and f in test.columns]

print("Features used:", FEATURES)

In [ ]:
GROUP_COLS = ["city", "station_name"]


def impute_within_groups(df):
    """Interpolate short gaps inside each station time series."""
    parts = []
    for keys, g in df.groupby(GROUP_COLS, dropna=False):
        g = g.sort_values("datetime").copy()
        g = g.set_index("datetime")

        numeric_cols = [c for c in FEATURES if c in g.columns]
        g[numeric_cols] = g[numeric_cols].interpolate(
            method="time", limit=6, limit_direction="both"
        )
        g = g.dropna(subset=["pm25"])

        # Restore group identifier columns
        for i, col in enumerate(GROUP_COLS):
            if col in g.columns:
                g[col] = g[col].iloc[0]
            else:
                g[col] = keys[i] if isinstance(keys, tuple) else keys

        parts.append(g.reset_index())

    return pd.concat(parts, ignore_index=True)


train_p = impute_within_groups(train)
test_p  = impute_within_groups(test)

print("Prepared train:", train_p.shape)
print("Prepared test :", test_p.shape)

In [ ]:
# Physically implausible PM2.5 -> NaN, then re-impute short gaps
def apply_limits(df):
    df = df.copy()
    df.loc[(df["pm25"] < 0) | (df["pm25"] > 1000), "pm25"] = np.nan
    return df

train_p = apply_limits(train_p)
test_p  = apply_limits(test_p)

# Re-impute short gaps after removing invalid values
train_p = impute_within_groups(train_p)
test_p  = impute_within_groups(test_p)

In [ ]:
# Fit scalers ONLY on training-city data.
feature_scaler = StandardScaler()
target_scaler  = StandardScaler()

train_p[FEATURES] = feature_scaler.fit_transform(train_p[FEATURES])
test_p[FEATURES]  = feature_scaler.transform(test_p[FEATURES])

target_scaler.fit(train_p[["pm25"]])

joblib.dump(feature_scaler, "feature_scaler.pkl")
joblib.dump(target_scaler,  "target_scaler.pkl")

pm25_idx = FEATURES.index("pm25")
print("Scalers fitted using TRAINING CITIES ONLY.")
print(f"PM2.5 mean={feature_scaler.mean_[pm25_idx]:.2f}  std={feature_scaler.scale_[pm25_idx]:.2f}")

In [ ]:
def make_sequences(df, lookback=24, horizon=1):
    """Build (X, y) sequences grouped by city+station so time series are not mixed."""
    X, y, city_labels = [], [], []

    for (city, station), g in df.groupby(GROUP_COLS):
        g = g.sort_values("datetime").reset_index(drop=True)

        values = g[FEATURES].values.astype(np.float32)
        target = g["pm25"].values.astype(np.float32)

        for i in range(lookback, len(g) - horizon + 1):
            end = i + horizon - 1
            seq = values[i - lookback : i]
            target_val = target[end]

            if np.isfinite(seq).all() and np.isfinite(target_val):
                X.append(seq)
                y.append(target_val)
                city_labels.append(city)

    return np.array(X), np.array(y), np.array(city_labels)


X_train, y_train, train_city_labels = make_sequences(train_p, LOOKBACK, HORIZON)
X_test,  y_test,  test_city_labels  = make_sequences(test_p,  LOOKBACK, HORIZON)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

print("\nTest sequences by city:")
print(pd.Series(test_city_labels).value_counts())

In [ ]:
# Inverse-transform helpers
pm25_idx   = FEATURES.index("pm25")
pm25_mean  = feature_scaler.mean_[pm25_idx]
pm25_scale = feature_scaler.scale_[pm25_idx]


def inverse_pm25(y_scaled):
    """Convert scaled PM2.5 back to original ug/m3 units."""
    return np.asarray(y_scaled) * pm25_scale + pm25_mean


# Alias kept for backwards compatibility with the rest of the notebook
def inverse_pm25_from_feature_scaler(y_scaled):
    return inverse_pm25(y_scaled)

# Deep-learning models

We use the same input sequences and the same test cities for every neural model.

- **LSTM:** temporal baseline
- **GRU:** lighter recurrent baseline
- **CNN-LSTM:** hybrid model that extracts local temporal patterns with 1-D CNN and then learns longer dependencies with LSTM


In [ ]:
n_features = X_train.shape[2]

def build_lstm(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def build_gru(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        GRU(64),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

def build_cnn_lstm(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        Conv1D(64, kernel_size=3, activation="relu", padding="same"),
        Conv1D(64, kernel_size=3, activation="relu", padding="same"),
        MaxPooling1D(pool_size=2),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6
    )
]

input_shape = (LOOKBACK, n_features)

lstm_model = build_lstm(input_shape)
gru_model = build_gru(input_shape)
cnn_lstm_model = build_cnn_lstm(input_shape)

lstm_model.summary()


In [ ]:
# Train/validation split INSIDE TRAINING CITIES only.
# The validation set is the latest 20% of sequences from each training city.
# No Bengaluru/Hyderabad data is used here.

def city_time_split(X, y, labels, val_fraction=0.2):
    train_idx, val_idx = [], []

    for city in np.unique(labels):
        idx = np.where(labels == city)[0]
        cut = int(len(idx) * (1 - val_fraction))
        train_idx.extend(idx[:cut])
        val_idx.extend(idx[cut:])

    return (
        X[train_idx], y[train_idx],
        X[val_idx], y[val_idx]
    )

X_tr, y_tr, X_val, y_val = city_time_split(
    X_train, y_train, train_city_labels
)

print(X_tr.shape, y_tr.shape)
print(X_val.shape, y_val.shape)


In [ ]:
history_lstm = lstm_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

history_gru = gru_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

history_cnn_lstm = cnn_lstm_model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Classical ML models require 2-D input.
# Flatten the 24-hour sequence into one feature vector.
X_tr_flat = X_tr.reshape(len(X_tr), -1)
X_val_flat = X_val.reshape(len(X_val), -1)
X_test_flat = X_test.reshape(len(X_test), -1)

lr_model = LinearRegression()
rf_model = RandomForestRegressor(
    n_estimators=250,
    max_depth=18,
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1
)
xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=SEED,
    n_jobs=-1
)

print("Training Linear Regression...")
lr_model.fit(X_tr_flat, y_tr)

print("Training Random Forest...")
rf_model.fit(X_tr_flat, y_tr)

print("Training XGBoost...")
xgb_model.fit(X_tr_flat, y_tr)

joblib.dump(lr_model, "linear_regression.pkl")
joblib.dump(rf_model, "random_forest.pkl")
joblib.dump(xgb_model, "xgboost.pkl")


In [ ]:
# Prediction helpers

def evaluate_predictions(y_true_scaled, y_pred_scaled):
    y_true = inverse_pm25_from_feature_scaler(y_true_scaled)
    y_pred = inverse_pm25_from_feature_scaler(y_pred_scaled)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    # MAPE is unstable when actual PM2.5 is near zero.
    mask = np.abs(y_true) > 1.0
    if mask.sum() > 0:
        mape = np.mean(
            np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
        ) * 100
    else:
        mape = np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_%": mape
    }

predictions = {}

predictions["Linear Regression"] = lr_model.predict(X_test_flat)
predictions["Random Forest"] = rf_model.predict(X_test_flat)
predictions["XGBoost"] = xgb_model.predict(X_test_flat)

predictions["LSTM"] = lstm_model.predict(X_test, verbose=0).ravel()
predictions["GRU"] = gru_model.predict(X_test, verbose=0).ravel()
predictions["CNN-LSTM"] = cnn_lstm_model.predict(X_test, verbose=0).ravel()

results = []

for model_name, pred in predictions.items():
    row = evaluate_predictions(y_test, pred)
    row["Model"] = model_name
    results.append(row)

results_df = pd.DataFrame(results)[
    ["Model", "MAE", "RMSE", "R2", "MAPE_%"]
].sort_values("RMSE")

results_df


In [ ]:
# Evaluate separately on each unseen city

city_results = []

for city in TEST_CITIES:
    mask = test_city_labels == city

    for model_name, pred in predictions.items():
        row = evaluate_predictions(y_test[mask], pred[mask])
        row["City"] = city
        row["Model"] = model_name
        city_results.append(row)

city_results_df = pd.DataFrame(city_results)[
    ["City", "Model", "MAE", "RMSE", "R2", "MAPE_%"]
].sort_values(["City", "RMSE"])

city_results_df


In [ ]:
# Plot RMSE by unseen city
plt.figure(figsize=(12, 6))
sns.barplot(data=city_results_df, x="City", y="RMSE", hue="Model")
plt.title("Cross-City PM2.5 Forecasting — RMSE on Unseen Cities")
plt.ylabel("RMSE (µg/m³)")
plt.xlabel("Unseen test city")
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Plot R² by unseen city
plt.figure(figsize=(12, 6))
sns.barplot(data=city_results_df, x="City", y="R2", hue="Model")
plt.title("Cross-City PM2.5 Forecasting — R² on Unseen Cities")
plt.ylabel("R²")
plt.xlabel("Unseen test city")
plt.tight_layout()
plt.show()


In [ ]:
# Detailed prediction plot for the best model on each unseen city

best_model = results_df.iloc[0]["Model"]
print("Best overall unseen-city model:", best_model)

for city in TEST_CITIES:
    mask = test_city_labels == city
    city_idx = np.where(mask)[0]

    if len(city_idx) == 0:
        continue

    pred = predictions[best_model][city_idx]
    actual = y_test[city_idx]

    actual = inverse_pm25_from_feature_scaler(actual)
    pred = inverse_pm25_from_feature_scaler(pred)

    n_plot = min(500, len(actual))

    plt.figure(figsize=(14, 5))
    plt.plot(actual[:n_plot], label="Actual")
    plt.plot(pred[:n_plot], label="Predicted")
    plt.title(f"{best_model}: PM2.5 Prediction — Unseen City: {city}")
    plt.xlabel("Test sequence")
    plt.ylabel("PM2.5 (µg/m³)")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Save the neural models
lstm_model.save("lstm_pm25.keras")
gru_model.save("gru_pm25.keras")
cnn_lstm_model.save("cnn_lstm_pm25.keras")

# Save final results
results_df.to_csv("overall_unseen_city_results.csv", index=False)
city_results_df.to_csv("per_city_unseen_results.csv", index=False)

print("Saved models, scalers and result tables.")


# How to report the experiment

Your main claim should be based on the **unseen-city test results**, not on training accuracy.

Recommended table:

| Model | Bengaluru MAE | Bengaluru RMSE | Bengaluru R² | Hyderabad MAE | Hyderabad RMSE | Hyderabad R² |
|---|---:|---:|---:|---:|---:|---:|
| Linear Regression | | | | | | |
| Random Forest | | | | | | |
| XGBoost | | | | | | |
| LSTM | | | | | | |
| GRU | | | | | | |
| **CNN-LSTM** | | | | | | |

### Important research rule

Do **not** tune the model using Bengaluru or Hyderabad results. They are your unseen test cities.

If you later want a stronger experiment, repeat the entire pipeline as **Leave-One-City-Out (LOCO)**:

> Train on 5 cities → test on the 6th city, repeated six times.

That is a stronger measure of cross-city generalization than a single train/test city split.
